# Pinecone — SDK v10 + LangChain Core 직접 통합

이 노트북은 책의 Pinecone 예제를 **2026-09-21 기준 공식 SDK 패턴**으로 다시 구성한 것입니다.

현재 `pinecone` SDK는 v10 계열이지만, 기존 `langchain-pinecone` 릴리스는 더 오래된 SDK 범위를 요구합니다. 또한 `langchain-community`는 2026년 6월 종료되었습니다. 따라서 이 노트북은 **Pinecone SDK를 직접 사용하고 `langchain-core`의 `BaseRetriever`만 얇게 연결**합니다. SDK 세대가 섞이는 문제를 피하면서 요청·응답 구조도 그대로 배울 수 있습니다.

학습 목표:

- PDF 로드·분할·metadata 정규화
- v10 typed schema로 Managed dense index 생성, upsert, 필터 검색, 삭제
- namespace와 명시적 ID를 이용한 재실행 안전성
- 한국어 형태소 기반 BM25 sparse + dense 하이브리드 검색
- Pinecone hosted reranker를 이용한 2단계 검색

참고:

- [Pinecone Python SDK](https://docs.pinecone.io/reference/sdks/python/overview)
- [Pinecone index 생성](https://docs.pinecone.io/guides/index-data/create-an-index)
- [Pinecone hybrid search](https://docs.pinecone.io/guides/search/hybrid-search)
- [Pinecone rerank](https://docs.pinecone.io/guides/search/rerank-results)


## 1. 설치

`pinecone-client`는 옛 패키지명입니다. 새 환경에서 `pinecone` v10과 LangChain Core 1.x를 설치합니다. major 범위를 고정해 향후 파괴적 변경이 실습 코드를 갑자기 바꾸지 않게 합니다.


In [ ]:
%pip install -qU           "pinecone>=10,<11" "langchain-core>=1,<2"           "langchain-openai>=1,<2" "langchain-text-splitters>=1,<2"           pymupdf kiwipiepy mmh3 numpy python-dotenv


## 2. 환경 변수

`.env` 예시:

```dotenv
OPENAI_API_KEY="..."
PINECONE_API_KEY="..."
PINECONE_INDEX_NAME="teddynote-rag-modern"          # 선택
PINECONE_HYBRID_INDEX_NAME="teddynote-hybrid-modern" # 선택
PINECONE_CLOUD="aws"                                 # 선택
PINECONE_REGION="us-east-1"                         # 선택
LANGSMITH_API_KEY="..."                             # 선택
```


In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

if os.getenv("LANGSMITH_API_KEY"):
    os.environ.setdefault("LANGSMITH_TRACING", "true")
    os.environ.setdefault("LANGSMITH_PROJECT", "vectorstores-pinecone-modern")

required_keys = ["OPENAI_API_KEY", "PINECONE_API_KEY"]
missing_keys = [key for key in required_keys if not os.getenv(key)]
if missing_keys:
    raise EnvironmentError(f"필수 환경 변수가 없습니다: {', '.join(missing_keys)}")


In [ ]:
from importlib.metadata import version

for package in ("pinecone", "langchain-core", "langchain-openai", "pymupdf"):
    print(f"{package}: {version(package)}")


## 3. PDF 로드·분할·metadata 정규화

`langchain-community` 로더 대신 PyMuPDF를 직접 사용합니다. Pinecone metadata에는 `None`이나 임의의 중첩 객체 대신 문자열·숫자·불리언·문자열 목록처럼 지원되는 값만 넣습니다. 페이지 번호는 사람이 읽기 쉬운 1부터 시작합니다.


In [ ]:
from pathlib import Path

import pymupdf
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

pdf_paths = sorted(Path("data").glob("*.pdf"))

if pdf_paths:
    raw_documents = []
    for path in pdf_paths:
        with pymupdf.open(path) as pdf:
            author = str(pdf.metadata.get("author") or "unknown")
            for page_number, page in enumerate(pdf, start=1):
                raw_documents.append(
                    Document(
                        page_content=page.get_text("text"),
                        metadata={
                            "source": path.name,
                            "page": page_number,
                            "author": author,
                        },
                    )
                )
else:
    raw_documents = [
        Document(
            page_content="GPT-4o mini는 비용 효율적인 소형 멀티모달 모델로 소개되었다.",
            metadata={"source": "ai-brief.pdf", "page": 1, "author": "sample"},
        ),
        Document(
            page_content="Anthropic은 Claude 3.5 Sonnet을 공개하며 코딩과 시각 추론 성능을 강조했다.",
            metadata={"source": "ai-brief.pdf", "page": 2, "author": "sample"},
        ),
        Document(
            page_content="벡터 검색은 임베딩 공간에서 의미가 가까운 문서를 찾는다.",
            metadata={"source": "rag-guide.pdf", "page": 3, "author": "sample"},
        ),
        Document(
            page_content="하이브리드 검색은 dense 의미 검색과 sparse 키워드 검색을 결합한다.",
            metadata={"source": "rag-guide.pdf", "page": 4, "author": "sample"},
        ),
        Document(
            page_content="Reranking은 1차 검색 후보를 더 정교한 모델로 다시 정렬한다.",
            metadata={"source": "rag-guide.pdf", "page": 5, "author": "sample"},
        ),
    ]

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=80)
split_documents = splitter.split_documents(raw_documents)


def normalize_document(document: Document, *, min_length: int = 5) -> Document | None:
    text = " ".join(document.page_content.split())
    if len(text) < min_length:
        return None
    metadata = {
        "source": Path(str(document.metadata.get("source", "unknown"))).name,
        "page": int(document.metadata.get("page", 0)),
        "author": str(document.metadata.get("author") or "unknown"),
    }
    return Document(page_content=text, metadata=metadata)


documents = [
    normalized
    for document in split_documents
    if (normalized := normalize_document(document)) is not None
]

print(f"PDF 수: {len(pdf_paths)}, 문서 조각 수: {len(documents)}")
documents[:2]


## 4. Dense Managed index 생성

`text-embedding-3-small`의 차원을 1536으로 명시하고 index도 같은 차원으로 만듭니다. v10의 새 코드에서는 예전 `dimension=`/`metric=`/`ServerlessSpec` 축약형 대신 **typed `schema` + `deployment`**를 사용합니다.

Vectors API의 `values`/`sparse_values` 레코드는 예약 필드 `_values`/`_sparse_values`에 대응합니다. 기존 index를 재사용할 때도 schema를 검사하고, 데이터 작업은 `describe()`가 돌려준 고유 host에 연결합니다.


In [ ]:
from langchain_openai import OpenAIEmbeddings
from pinecone import Pinecone, SchemaBuilder

embedding_dimension = 1536
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    dimensions=embedding_dimension,
)

pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])
index_name = os.getenv("PINECONE_INDEX_NAME", "teddynote-rag-modern")
cloud = os.getenv("PINECONE_CLOUD", "aws")
region = os.getenv("PINECONE_REGION", "us-east-1")

dense_field = "_values"
dense_schema = (
    SchemaBuilder()
    .add_dense_vector_field(
        dense_field,
        dimension=embedding_dimension,
        metric="cosine",
    )
    .build()
)
deployment = {
    "deployment_type": "managed",
    "cloud": cloud,
    "region": region,
}

if not pc.indexes.exists(index_name):
    pc.indexes.create(
        name=index_name,
        schema=dense_schema,
        deployment=deployment,
        deletion_protection="disabled",
        tags={"purpose": "rag-study-dense"},
    )

description = pc.indexes.describe(index_name)
existing_dense_field = description.schema.fields.get(dense_field)
if (
    existing_dense_field is None
    or existing_dense_field.dimension != embedding_dimension
    or existing_dense_field.metric != "cosine"
):
    raise ValueError(
        f"기존 index의 {dense_field!r} schema가 기대한 dense 설정과 다릅니다."
    )

index = pc.index(host=description.host)
description


## 5. Pinecone SDK용 dense 어댑터

본문은 metadata의 `_text`에 함께 저장합니다. Upsert용 ID를 명시하면 같은 셀을 다시 실행해도 같은 레코드를 갱신하므로 중복되지 않습니다. `BaseRetriever` 어댑터는 LCEL에서 표준 `invoke()`/`ainvoke()` 인터페이스를 제공합니다.


In [ ]:
from collections.abc import Sequence
from typing import Any

from langchain_core.embeddings import Embeddings
from langchain_core.retrievers import BaseRetriever
from pydantic import ConfigDict

TEXT_KEY = "_text"


def pinecone_upsert_documents(
    index: Any,
    documents: Sequence[Document],
    ids: Sequence[str],
    embeddings: Embeddings,
    namespace: str,
    *,
    batch_size: int = 64,
    text_key: str = TEXT_KEY,
) -> list[str]:
    if len(documents) != len(ids):
        raise ValueError("documents와 ids 길이가 같아야 합니다.")
    if len(set(ids)) != len(ids):
        raise ValueError("ids는 고유해야 합니다.")
    if not documents:
        return []

    vectors = embeddings.embed_documents([document.page_content for document in documents])
    records = [
        {
            "id": id_,
            "values": vector,
            "metadata": {**document.metadata, text_key: document.page_content},
        }
        for id_, document, vector in zip(ids, documents, vectors)
    ]
    response = index.upsert(
        vectors=records,
        namespace=namespace,
        batch_size=batch_size,
        show_progress=False,
    )
    if response.has_errors:
        raise RuntimeError(
            f"일부 upsert 실패: {response.failed_item_count}/{response.total_item_count}"
        )
    return list(ids)


def pinecone_similarity_search_with_score(
    index: Any,
    embeddings: Embeddings,
    query: str,
    namespace: str,
    *,
    k: int = 4,
    filter: dict | None = None,
    text_key: str = TEXT_KEY,
) -> list[tuple[Document, float]]:
    response = index.query(
        vector=embeddings.embed_query(query),
        top_k=k,
        include_metadata=True,
        namespace=namespace,
        filter=filter,
    )
    results = []
    for match in response.matches:
        metadata = dict(match.metadata or {})
        text = str(metadata.pop(text_key, ""))
        results.append(
            (
                Document(id=str(match.id), page_content=text, metadata=metadata),
                float(match.score),
            )
        )
    return results


class PineconeDenseRetriever(BaseRetriever):
    embeddings: Embeddings
    index: Any
    namespace: str
    top_k: int = 4
    metadata_filter: dict | None = None
    text_key: str = TEXT_KEY

    model_config = ConfigDict(arbitrary_types_allowed=True)

    def _get_relevant_documents(self, query: str, *, run_manager, **kwargs) -> list[Document]:
        top_k = int(kwargs.get("top_k", self.top_k))
        filter_ = kwargs.get("filter", self.metadata_filter)
        scored = pinecone_similarity_search_with_score(
            index=self.index,
            embeddings=self.embeddings,
            query=query,
            namespace=self.namespace,
            k=top_k,
            filter=filter_,
            text_key=self.text_key,
        )
        return [
            Document(
                id=document.id,
                page_content=document.page_content,
                metadata={**document.metadata, "retrieval_score": score},
            )
            for document, score in scored
        ]


## 6. Upsert, 검색, 점수, metadata 필터


In [ ]:
namespace = "teddynote-dense-v1"
document_ids = [f"dense-{i:05d}" for i in range(len(documents))]
upserted_ids = pinecone_upsert_documents(
    index=index,
    documents=documents,
    ids=document_ids,
    embeddings=embeddings,
    namespace=namespace,
)
upserted_ids[:3], index.describe_index_stats()


Pinecone은 eventually consistent하므로 upsert 직후 통계나 검색 결과 반영에 짧은 지연이 있을 수 있습니다. 실서비스에서는 고정 `sleep`보다 제한된 재시도와 backoff를 사용하세요.


In [ ]:
query = "GPT-4o mini 출시와 관련된 내용을 알려주세요"
scored_results = pinecone_similarity_search_with_score(
    index,
    embeddings,
    query,
    namespace,
    k=3,
)
for document, score in scored_results:
    print(f"score={score:.4f}", document.metadata, "\n", document.page_content, "\n")


In [ ]:
source_to_search = documents[0].metadata["source"]
filtered_results = pinecone_similarity_search_with_score(
    index,
    embeddings,
    "AI 모델 발표",
    namespace,
    k=3,
    filter={"source": {"$eq": source_to_search}},
)
filtered_results


## 7. LangChain Retriever


In [ ]:
retriever = PineconeDenseRetriever(
    embeddings=embeddings,
    index=index,
    namespace=namespace,
    top_k=3,
)
retriever.invoke("Claude 3.5 Sonnet 관련 내용을 알려주세요")


In [ ]:
filtered_retriever = PineconeDenseRetriever(
    embeddings=embeddings,
    index=index,
    namespace=namespace,
    top_k=2,
    metadata_filter={"page": {"$lte": 3}},
)
filtered_retriever.invoke("AI 모델 발표")


## 8. 추가·삭제와 namespace 정리

ID, metadata filter, namespace 전체 삭제는 SDK의 `index.delete()`를 사용합니다. 전체 삭제 예제는 사고 방지를 위해 기본적으로 꺼 둡니다.


In [ ]:
scratch_id = "scratch-delete-001"
pinecone_upsert_documents(
    index=index,
    documents=[
        Document(
            page_content="삭제 동작을 확인하기 위한 임시 문서입니다.",
            metadata={"source": "scratch.txt", "page": 0, "author": "sample"},
        )
    ],
    ids=[scratch_id],
    embeddings=embeddings,
    namespace=namespace,
)
index.delete(ids=[scratch_id], namespace=namespace)


In [ ]:
RUN_DESTRUCTIVE_CLEANUP = False

if RUN_DESTRUCTIVE_CLEANUP:
    index.delete(
        filter={"source": {"$eq": "scratch.txt"}},
        namespace=namespace,
    )
    # namespace 전체를 지우려면 대상을 재확인한 뒤 다음 줄을 사용합니다.
    # index.delete(delete_all=True, namespace=namespace)


## 9. Dense + sparse 하이브리드 검색

기존 Vectors API 워크로드에서는 한 레코드에 dense와 sparse 벡터를 함께 저장하고 `alpha`로 두 신호의 비중을 조절할 수 있습니다. 이 단일-index 방식은 dense 필드에 `dotproduct` metric을 사용하며, v10 schema에는 sparse 필드도 처음부터 명시해야 합니다.

범용 `pinecone-text`의 영어 중심 기본 tokenizer 대신 Kiwi 형태소 분석기로 작은 한국어 BM25 encoder를 구현합니다. 운영 환경에서는 학습 corpus·불용어·형태소 품사·해시 충돌을 자체 평가하고 encoder JSON을 index 버전과 함께 관리해야 합니다.


In [ ]:
hybrid_index_name = os.getenv(
    "PINECONE_HYBRID_INDEX_NAME",
    "teddynote-hybrid-modern",
)

hybrid_dense_field = "_values"
hybrid_sparse_field = "_sparse_values"
hybrid_schema = (
    SchemaBuilder()
    .add_dense_vector_field(
        hybrid_dense_field,
        dimension=embedding_dimension,
        metric="dotproduct",
    )
    .add_sparse_vector_field(hybrid_sparse_field)
    .build()
)

if not pc.indexes.exists(hybrid_index_name):
    pc.indexes.create(
        name=hybrid_index_name,
        schema=hybrid_schema,
        deployment=deployment,
        deletion_protection="disabled",
        tags={"purpose": "rag-study-hybrid"},
    )

hybrid_description = pc.indexes.describe(hybrid_index_name)
existing_hybrid_dense = hybrid_description.schema.fields.get(hybrid_dense_field)
if (
    existing_hybrid_dense is None
    or existing_hybrid_dense.dimension != embedding_dimension
    or existing_hybrid_dense.metric != "dotproduct"
    or hybrid_sparse_field not in hybrid_description.schema.fields
):
    raise ValueError(
        "하이브리드 index에는 dot-product dense 필드와 sparse 필드가 모두 필요합니다."
    )
hybrid_index = pc.index(host=hybrid_description.host)


In [ ]:
import json
import math
from collections import Counter

import mmh3
from kiwipiepy import Kiwi


class KiwiBM25Encoder:
    '''Kiwi 토큰을 Pinecone sparse vector로 바꾸는 학습용 BM25 encoder.'''

    allowed_tag_prefixes = ("N", "V", "MA", "SL", "SN", "XR")

    def __init__(
        self,
        *,
        k1: float = 1.5,
        b: float = 0.75,
        stopwords: set[str] | None = None,
    ):
        self.k1 = k1
        self.b = b
        self.stopwords = stopwords or {"것", "수", "등", "및", "더", "이", "그"}
        self.kiwi = Kiwi()
        self.idf: dict[str, float] = {}
        self.avgdl = 0.0
        self.n_docs = 0

    def tokenize(self, text: str) -> list[str]:
        return [
            token.form.lower()
            for token in self.kiwi.tokenize(text, normalize_coda=True)
            if token.tag.startswith(self.allowed_tag_prefixes)
            and token.form.lower() not in self.stopwords
        ]

    def fit(self, corpus: Sequence[str]) -> "KiwiBM25Encoder":
        tokenized = [self.tokenize(text) for text in corpus]
        if not tokenized:
            raise ValueError("corpus는 비어 있을 수 없습니다.")
        self.n_docs = len(tokenized)
        self.avgdl = sum(map(len, tokenized)) / self.n_docs
        document_frequency = Counter(
            token for tokens in tokenized for token in set(tokens)
        )
        self.idf = {
            token: math.log(1.0 + (self.n_docs - df + 0.5) / (df + 0.5))
            for token, df in document_frequency.items()
        }
        return self

    @staticmethod
    def _hashed_sparse(weights: dict[str, float]) -> dict[str, list]:
        by_index: dict[int, float] = {}
        for token, weight in weights.items():
            index = mmh3.hash(token, signed=False)
            by_index[index] = by_index.get(index, 0.0) + float(weight)
        ordered = sorted(by_index.items())
        return {
            "indices": [index for index, _ in ordered],
            "values": [value for _, value in ordered],
        }

    def encode_documents(self, texts: Sequence[str]) -> list[dict[str, list]]:
        if not self.idf or self.avgdl <= 0:
            raise RuntimeError("먼저 fit(corpus)을 호출하세요.")
        encoded = []
        for text in texts:
            tokens = self.tokenize(text)
            counts = Counter(tokens)
            length = len(tokens)
            weights = {
                token: (
                    count * (self.k1 + 1.0)
                    / (
                        count
                        + self.k1
                        * (1.0 - self.b + self.b * length / self.avgdl)
                    )
                )
                for token, count in counts.items()
                if token in self.idf
            }
            encoded.append(self._hashed_sparse(weights))
        return encoded

    def encode_queries(self, texts: Sequence[str]) -> list[dict[str, list]]:
        if not self.idf:
            raise RuntimeError("먼저 fit(corpus)을 호출하세요.")
        return [
            self._hashed_sparse(
                {token: self.idf[token] for token in set(self.tokenize(text)) if token in self.idf}
            )
            for text in texts
        ]

    def dump(self, path: str | Path) -> None:
        payload = {
            "k1": self.k1,
            "b": self.b,
            "stopwords": sorted(self.stopwords),
            "idf": self.idf,
            "avgdl": self.avgdl,
            "n_docs": self.n_docs,
        }
        Path(path).write_text(
            json.dumps(payload, ensure_ascii=False, indent=2),
            encoding="utf-8",
        )

    @classmethod
    def load(cls, path: str | Path) -> "KiwiBM25Encoder":
        payload = json.loads(Path(path).read_text(encoding="utf-8"))
        encoder = cls(
            k1=float(payload["k1"]),
            b=float(payload["b"]),
            stopwords=set(payload["stopwords"]),
        )
        encoder.idf = {token: float(value) for token, value in payload["idf"].items()}
        encoder.avgdl = float(payload["avgdl"])
        encoder.n_docs = int(payload["n_docs"])
        return encoder


In [ ]:
def hybrid_scale(
    dense: Sequence[float],
    sparse: dict[str, list],
    alpha: float,
) -> tuple[list[float], dict[str, list]]:
    if not 0.0 <= alpha <= 1.0:
        raise ValueError("alpha는 0과 1 사이여야 합니다.")
    dense_scaled = [float(value) * alpha for value in dense]
    sparse_pairs = [
        (index, float(value) * (1.0 - alpha))
        for index, value in zip(sparse["indices"], sparse["values"])
        if value != 0 and alpha < 1.0
    ]
    return dense_scaled, {
        "indices": [index for index, _ in sparse_pairs],
        "values": [value for _, value in sparse_pairs],
    }


class PineconeHybridRetriever(BaseRetriever):
    embeddings: Embeddings
    sparse_encoder: KiwiBM25Encoder
    index: Any
    namespace: str
    top_k: int = 4
    alpha: float = 0.5
    text_key: str = TEXT_KEY

    model_config = ConfigDict(arbitrary_types_allowed=True)

    def add_documents(
        self,
        documents: Sequence[Document],
        ids: Sequence[str],
        *,
        batch_size: int = 64,
    ) -> list[str]:
        if len(documents) != len(ids):
            raise ValueError("documents와 ids 길이가 같아야 합니다.")
        if len(set(ids)) != len(ids):
            raise ValueError("ids는 고유해야 합니다.")
        if not documents:
            return []

        texts = [document.page_content for document in documents]
        dense_vectors = self.embeddings.embed_documents(texts)
        sparse_vectors = self.sparse_encoder.encode_documents(texts)
        records = []
        for id_, document, dense, sparse in zip(
            ids, documents, dense_vectors, sparse_vectors
        ):
            record = {
                "id": id_,
                "values": dense,
                "metadata": {
                    **document.metadata,
                    self.text_key: document.page_content,
                },
            }
            if sparse["indices"]:
                record["sparse_values"] = sparse
            records.append(record)
        response = self.index.upsert(
            vectors=records,
            namespace=self.namespace,
            batch_size=batch_size,
            show_progress=False,
        )
        if response.has_errors:
            raise RuntimeError(
                f"일부 upsert 실패: {response.failed_item_count}/{response.total_item_count}"
            )
        return list(ids)

    def _get_relevant_documents(self, query: str, *, run_manager, **kwargs) -> list[Document]:
        alpha = float(kwargs.get("alpha", self.alpha))
        top_k = int(kwargs.get("top_k", self.top_k))
        filter_ = kwargs.get("filter")
        dense = self.embeddings.embed_query(query)
        sparse = self.sparse_encoder.encode_queries([query])[0]
        dense_scaled, sparse_scaled = hybrid_scale(dense, sparse, alpha)

        query_kwargs = {
            "vector": dense_scaled,
            "top_k": top_k,
            "include_metadata": True,
            "namespace": self.namespace,
            "filter": filter_,
        }
        if sparse_scaled["indices"]:
            query_kwargs["sparse_vector"] = sparse_scaled
        response = self.index.query(**query_kwargs)

        results = []
        for match in response.matches:
            metadata = dict(match.metadata or {})
            text = str(metadata.pop(self.text_key, ""))
            metadata["retrieval_score"] = float(match.score)
            results.append(
                Document(id=str(match.id), page_content=text, metadata=metadata)
            )
        return results


In [ ]:
corpus = [document.page_content for document in documents]
sparse_encoder = KiwiBM25Encoder().fit(corpus)

sparse_encoder_path = Path("kiwi_bm25_encoder.json")
sparse_encoder.dump(sparse_encoder_path)
sparse_encoder = KiwiBM25Encoder.load(sparse_encoder_path)

hybrid_namespace = "teddynote-hybrid-v1"
hybrid_retriever = PineconeHybridRetriever(
    embeddings=embeddings,
    sparse_encoder=sparse_encoder,
    index=hybrid_index,
    namespace=hybrid_namespace,
    top_k=3,
    alpha=0.5,
)
hybrid_ids = [f"hybrid-{i:05d}" for i in range(len(documents))]
hybrid_retriever.add_documents(documents, hybrid_ids)


In [ ]:
hybrid_results = hybrid_retriever.invoke("앤스로픽 Claude 3.5 출시")
for document in hybrid_results:
    print(document.metadata, "\n", document.page_content, "\n")


`alpha=1.0`은 dense만, `alpha=0.0`은 sparse만 사용합니다. 중간값은 두 점수 신호를 선형 결합합니다. alpha는 데이터셋과 평가 지표로 조정해야 합니다.


In [ ]:
query = "Anthropic Claude 3.5"
dense_only = hybrid_retriever.invoke(query, alpha=1.0, top_k=2)
sparse_only = hybrid_retriever.invoke(query, alpha=0.0, top_k=2)

print("dense only:", [doc.page_content for doc in dense_only])
print("sparse only:", [doc.page_content for doc in sparse_only])


In [ ]:
hybrid_filtered = hybrid_retriever.invoke(
    "AI 모델 발표",
    alpha=0.5,
    top_k=3,
    filter={"page": {"$lte": 3}},
)
hybrid_filtered


## 10. Pinecone hosted reranker

1차 dense 검색에서는 후보를 넉넉히 가져오고 `bge-reranker-v2-m3`로 다시 정렬합니다. rerank score는 클수록 관련성이 높습니다.


In [ ]:
rerank_query = "앤스로픽의 Claude 3.5 Sonnet 발표"
candidates_with_scores = pinecone_similarity_search_with_score(
    index,
    embeddings,
    rerank_query,
    namespace,
    k=min(8, len(documents)),
)
candidates = [document for document, _ in candidates_with_scores]

reranked = pc.inference.rerank(
    model="bge-reranker-v2-m3",
    query=rerank_query,
    documents=[
        {"id": document.id or str(position), "text": document.page_content}
        for position, document in enumerate(candidates)
    ],
    top_n=min(3, len(candidates)),
    return_documents=True,
    parameters={"truncate": "END"},
)
reranked


## 11. 정리와 선택 기준

- Pinecone v10 Vectors API를 직접 사용하고 LangChain에는 얇은 `BaseRetriever`로 연결
- dense 검색: cosine index + 외부 임베딩
- 기존 vector 워크로드의 hybrid: dot-product 단일 index에 dense+sparse 저장
- 새 텍스트 중심 워크로드: Pinecone의 integrated embedding/full-text 검색 기능도 함께 비교
- 품질 개선: 후보를 넓게 검색한 뒤 `pc.inference.rerank()`
- 운영: namespace로 격리하고, 스키마/임베딩/encoder 변경 시 새 버전으로 마이그레이션
- 안전: 전체 삭제는 플래그로 보호하고 대상 namespace/index를 다시 확인
